In [2]:
import asyncio
import json
import sys

class SimpleServer:
    def __init__(self, host='127.0.0.1', port=8889):
        self.host = host
        self.port = port
        self.clients = set()

    async def broadcast(self, sender_writer, message):
        """Send message to all clients except the sender."""
        payload = json.dumps({"sender": str(sender_writer.get_extra_info('peername')), "message": message}) + "\n"
        for client_writer in list(self.clients):
            if client_writer != sender_writer:
                try:
                    client_writer.write(payload.encode('utf-8'))
                    await client_writer.drain()
                except Exception:
                    # Remove broken connection
                    self.clients.discard(client_writer)

    async def handle_client(self, reader, writer):
        """Manage individual client connection lifecycle."""
        peer = writer.get_extra_info('peername')
        client_id = f"{peer[0]}:{peer[1]}"
        self.clients.add(writer)
        print(f"[+] Client connected: {client_id}")

        try:
            while True:
                data = await reader.readline()
                if not data:
                    break
                
                text = data.decode('utf-8').strip()
                print(f"[{client_id}]: {text}")
                await self.broadcast(writer, text)

        except (asyncio.CancelledError, ConnectionResetError):
            pass
        finally:
            self.clients.discard(writer)
            writer.close()
            await writer.wait_closed()
            print(f"[-] Client disconnected: {client_id}")

    async def start(self):
        """Starts the server with socket address reuse enabled."""
        try:
            # reuse_address=True prevents WinError 10048 / Errno 98 when restarting quickly
            server = await asyncio.start_server(
                self.handle_client, 
                self.host, 
                self.port,
                reuse_address=True
            )
            print("==================================================")
            print(f" Socket Server Listening on {self.host}:{self.port}")
            print(" Press Ctrl+C to stop the server safely.")
            print("==================================================")
            
            async with server:
                await server.serve_forever()

        except OSError as e:
            if getattr(e, 'winerror', None) == 10048 or getattr(e, 'errno', None) == 98:
                print(f"\n[ERROR] Port {self.port} is currently in use.")
                print("Close existing running scripts or change the port number.")
            else:
                print(f"\n[SOCKET ERROR]: {e}")


def main():
    server = SimpleServer(port=8889)
    
    try:
        # Detect whether an asyncio event loop is already active (e.g. Jupyter / Spyder)
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        # Running inside Jupyter, Spyder, or an active loop context
        print("[*] Active event loop detected (Jupyter/Interactive). Task scheduled.")
        loop.create_task(server.start())
    else:
        # Running as a standalone script from terminal
        try:
            asyncio.run(server.start())
        except KeyboardInterrupt:
            print("\n[!] Server manually stopped. Port released.")

if __name__ == '__main__':
    main()

[*] Active event loop detected (Jupyter/Interactive). Task scheduled.

[SOCKET ERROR]: [Errno 13] error while attempting to bind on address ('127.0.0.1', 8889): [winerror 10013] an attempt was made to access a socket in a way forbidden by its access permissions
